In [1]:
import selenium
from selenium import webdriver
import pandas as pd
from io import StringIO  # Import necessário para lidar com o HTML literal

url = 'https://fiis.com.br/lupa-de-fiis/'

driver = webdriver.Chrome()
driver.implicitly_wait(10)
driver.get(url)

# Capturando a tabela
tabela = driver.find_element("class name", "default-fiis-table")

# Corrigindo o erro com o uso de StringIO
html_string = tabela.get_attribute('innerHTML')
dados = pd.read_html(StringIO(html_string), encoding='utf-8')[0]

# Definindo as colunas conforme desejado
dados = pd.DataFrame(dados, columns = ['Ticker', 'Público Alvo','Tipo de Fii','Administrador','Data Pagamento', 'Data Base',
                                       'Último Rend. (R$)', 'Último Rend. (%)', 'Rend. Méd. 12m (R$)',
                                       'Rend. Méd. 12m (%)', 'Patrimônio/Cota', 'Cotação/VP', 'N° negócios/mês',
                                       'Partic. IFIX', 'Número Cotistas', 'Cota base', 'Patrimônio'])

# Organizando os dados
dados.sort_values('Ticker', inplace=True)

# Convertendo colunas para strings e lidando com valores nulos
col_floats = list(dados.iloc[:, 6:17].columns)
dados[col_floats] = dados[col_floats].astype(str).fillna('')

# Aplicando a substituição de caracteres
dados[col_floats] = dados[col_floats].apply(lambda x: x.str.replace('R$', '').str.replace('.0','').str.replace('.','').str.replace('%','').str.replace(',','.'))

# Convertendo para float
dados[col_floats] = dados[col_floats].apply(pd.to_numeric, errors='coerce')

# Ajustando colunas específicas
dados['Último Rend. (R$)'] = dados['Último Rend. (R$)']/100
dados['Último Rend. (%)'] = dados['Último Rend. (%)']/100
dados['Rend. Méd. 12m (R$)'] = dados['Rend. Méd. 12m (R$)']/100
dados['Rend. Méd. 12m (%)'] = dados['Rend. Méd. 12m (%)']/100
dados['Patrimônio/Cota'] = dados['Patrimônio/Cota']/100
dados['Cotação/VP'] = dados['Cotação/VP']/100
dados['Partic. IFIX'] = dados['Partic. IFIX']/100
dados['Cota base'] = dados['Cota base']/100

# Salvando para Excel
dados.to_excel(r'C:\ARQUIVOS\OneDrive\Bolsa\Analise\export_fiis.xlsx', index=None)

# Finalizando a sessão do WebDriver
driver.quit()
